# Exploration du bucket MinIO — Référentiel ANFA

Ce notebook se connecte au service MinIO (via le réseau Docker Compose),
liste les objets du bucket `anfa-raw` et analyse les données avec pandas.

> **Important** : l'endpoint est `http://minio:9000` (nom de service Docker),
> jamais `localhost` depuis un conteneur Jupyter.

## Cellule 1 — Installation de boto3

In [1]:
# L'image jupyter/scipy-notebook ne contient pas boto3 par défaut.
# On l'installe dans le kernel courant.
%pip install boto3==1.34.0

Note: you may need to restart the kernel to use updated packages.


## Cellule 2 — Connexion à MinIO

In [2]:
import boto3

# Depuis le conteneur Jupyter, MinIO est accessible par son nom de service
# dans le réseau Compose. Jamais localhost !
s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="anfa-app-key",
    aws_secret_access_key="anfa-app-secret-2026",
    region_name="us-east-1",
)

# Lister tous les buckets disponibles
reponse = s3.list_buckets()
print("Buckets disponibles :")
for bucket in reponse.get("Buckets", []):
    print(f"  - {bucket['Name']}")

Buckets disponibles :
  - anfa-raw


## Cellule 3 — Lister les objets du bucket

In [3]:
reponse = s3.list_objects_v2(Bucket="anfa-raw", Prefix="referentiel/")

print("Objets dans anfa-raw/referentiel/ :")
for obj in reponse.get("Contents", []):
    print(f"  {obj['Key']}  ({obj['Size']} octets)")

Objets dans anfa-raw/referentiel/ :
  referentiel/arrets.csv  (3821 octets)
  referentiel/bus.csv  (5912 octets)
  referentiel/lignes.csv  (945 octets)
  referentiel/tarifs.csv  (718 octets)


## Cellule 4 — Lire lignes.csv depuis MinIO avec pandas

In [4]:
import pandas as pd
from io import BytesIO

# Télécharger le CSV des lignes directement en mémoire (pas de fichier temporaire)
obj = s3.get_object(Bucket="anfa-raw", Key="referentiel/lignes.csv")
df_lignes = pd.read_csv(BytesIO(obj["Body"].read()))

print(f"Dimensions : {df_lignes.shape[0]} lignes × {df_lignes.shape[1]} colonnes")
df_lignes

Dimensions : 12 lignes × 6 colonnes


,ligne_id,nom,terminus_depart,terminus_arrivee,nb_arrets,distance_km
0,L01,Adidogomé - Adawlato,Adidogomé Assiyéyé,Adawlato Marché,9,9.89
1,L02,Agoè Assiyéyé - Grand Marché,Agoè Assiyéyé Terminus,Adawlato Grand Marché,9,16.28
2,L03,Avédji - Adawlato,Avédji Limousine,Adawlato Marché,9,11.67
3,L04,Université de Lomé - Adawlato,Campus Universitaire,Adawlato Marché,8,5.65
4,L05,Hédzranawoé - Adawlato,Hédzranawoé Terminus,Adawlato Marché,8,7.37
5,L06,Bè - Tokoin Casablanca,Bè Kpota,Tokoin Casablanca,7,5.49
6,L07,Cacavéli - BIA Centre,Cacavéli Marché,BIA Centre,7,10.03
7,L08,Aéroport GTA - Adawlato,Aéroport GTA,Adawlato Marché,7,8.73
8,L09,Baguida - Adawlato,Baguida Terminus,Adawlato Marché,8,13.13
9,L10,Anfamé - Université de Lomé,Anfamé Terminus,Campus Universitaire,8,5.89


## Cellule 5 — Analyse exploratoire

In [5]:
# Top 3 des lignes les plus longues
print("Top 3 des lignes les plus longues :")
df_lignes.nlargest(3, "distance_km")[["nom", "nb_arrets", "distance_km"]]

Top 3 des lignes les plus longues :


,nom,nb_arrets,distance_km
1,Agoè Assiyéyé - Grand Marché,9,16.28
8,Baguida - Adawlato,8,13.13
2,Avédji - Adawlato,9,11.67


## Cellule 6 — Lire et analyser tarifs.csv

In [6]:
obj_tarifs = s3.get_object(Bucket="anfa-raw", Key="referentiel/tarifs.csv")
df_tarifs = pd.read_csv(BytesIO(obj_tarifs["Body"].read()))

print("Tarif moyen par type :")
df_tarifs.groupby("type")["prix_fcfa"].mean().reset_index()

Tarif moyen par type :


,type,prix_fcfa
0,abonnement,14785.714286
1,unitaire,170.000000


## Cellule 7 — Lire bus.csv et compter les actifs

In [7]:
obj_bus = s3.get_object(Bucket="anfa-raw", Key="referentiel/bus.csv")
df_bus = pd.read_csv(BytesIO(obj_bus["Body"].read()))

print(f"Total bus     : {len(df_bus)}")
print(f"Bus actifs    : {len(df_bus[df_bus['statut'] == 'actif'])}")
print(f"Capacité totale : {df_bus['capacite'].sum()} places")

df_bus["statut"].value_counts()

Total bus     : 100
Bus actifs    : 93
Capacité totale : 4863 places


statut
actif           93
maintenance      6
hors_service     1
Name: count, dtype: int64